# Acquisition Initialization Workflow

multi-start acquisition optimizationでは、initial conditionsがどのbasinからlocal searchを始めるかを決めます。

```text
raw samples → acquisition-weighted selection → restart initial conditions
                                             ↓
                                      local optimization
                                             ↓
                                         candidate
```

通常q-batchとone-shot acquisitionではoptimization variableのshapeが異なるため、initialization contractも分けて考えます。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.knowledge_gradient import qKnowledgeGradient
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.fit import fit_gpytorch_mll
from botorch.optim.initializers import (
    gen_batch_initial_conditions,
    gen_one_shot_kg_initial_conditions,
)
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.models import SingleTaskGP
from robotorchan.optim import optimize_acqf

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

## 2. Fit the surrogate and construct qLogEI

In [ ]:
def response(X: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * X) + 0.2 * X


train_X = torch.linspace(0.05, 0.95, 9).unsqueeze(-1)
train_Y = response(train_X)
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(model.make_mll())
model.eval()

bounds = torch.tensor([[0.0], [1.0]])
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=101),
)

## 3. Standard q-batch initial conditions

通常のexplicit initial-condition shapeは`num_restarts x q x d`です。

In [ ]:
num_restarts = 4
q = 2
initial_conditions = gen_batch_initial_conditions(
    acq_function=acquisition,
    bounds=bounds,
    q=q,
    num_restarts=num_restarts,
    raw_samples=64,
)

assert initial_conditions.shape == torch.Size([num_restarts, q, 1])
print("initial-condition shape:", initial_conditions.shape)
print(initial_conditions)

## 4. Explicit starts are consumed by the optimizer

`batch_initial_conditions`を渡す場合、そのrestart setがlocal optimizationの開始点になります。

In [ ]:
candidate, value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=q,
    optimizer="botorch",
    num_restarts=num_restarts,
    raw_samples=None,
    batch_initial_conditions=initial_conditions,
)

assert candidate.shape == torch.Size([q, 1])
print("candidate:", candidate.squeeze(-1))
print("acquisition value:", value)

## 5. Fixed features participate in initialization

fixed coordinateはoptimizationだけでなくinitial-condition generationでも保持される必要があります。

In [ ]:
train_X_2d = torch.rand(12, 2)
train_Y_2d = (
    torch.sin(2 * torch.pi * train_X_2d[:, :1])
    + 0.25 * torch.cos(2 * torch.pi * train_X_2d[:, 1:2])
)
model_2d = SingleTaskGP(train_X_2d, train_Y_2d)
fit_gpytorch_mll(model_2d.make_mll())
model_2d.eval()
acq_2d = qLogExpectedImprovement(model=model_2d, best_f=train_Y_2d.max())
bounds_2d = torch.tensor([[0.0, 0.0], [1.0, 1.0]])

fixed_initial = gen_batch_initial_conditions(
    acq_function=acq_2d,
    bounds=bounds_2d,
    q=1,
    num_restarts=4,
    raw_samples=64,
    fixed_features={1: 0.4},
)

torch.testing.assert_close(fixed_initial[..., 1], torch.full((4, 1), 0.4))
print("fixed-feature starts:\n", fixed_initial)

## 6. Pending points do not add an initialization axis

`X_pending`はacquisition contextです。通常initial conditionsはpending rowsの数に関係なく`[restart, q, d]`のままです。

In [ ]:
X_pending = torch.tensor([[0.30], [0.70]])
pending_acq = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([64]), seed=202),
    X_pending=X_pending,
)
pending_initial = gen_batch_initial_conditions(
    acq_function=pending_acq,
    bounds=bounds,
    q=2,
    num_restarts=4,
    raw_samples=64,
)

assert pending_initial.shape == torch.Size([4, 2, 1])
print("pending-aware initial-condition shape:", pending_initial.shape)

## 7. Knowledge Gradient is a one-shot exception

qKGはreal candidateにfantasy/auxiliary decision rowsを加えたaugmented problemを最適化します。したがってordinary `[restart, q, d]` startsをそのまま使いません。

In [ ]:
kg = qKnowledgeGradient(
    model=model,
    num_fantasies=4,
)
public_q = 1
augmented_q = kg.get_augmented_q_batch_size(q=public_q)
kg_initial = gen_one_shot_kg_initial_conditions(
    acq_function=kg,
    bounds=bounds,
    q=public_q,
    num_restarts=3,
    raw_samples=32,
)

assert augmented_q == 5
assert kg_initial.shape == torch.Size([3, augmented_q, 1])
print("public q:", public_q)
print("augmented q:", augmented_q)
print("qKG initial-condition shape:", kg_initial.shape)

## 8. qKG optimization returns only public candidates

optimizer内部ではaugmented batchを扱いますが、one-shot acquisitionのcandidate extractionにより返されるreal candidateはrequested public qです。

In [ ]:
kg_candidate, kg_value = optimize_acqf(
    acq_function=kg,
    bounds=bounds,
    q=public_q,
    optimizer="botorch",
    num_restarts=3,
    raw_samples=32,
)

assert kg_candidate.shape == torch.Size([public_q, 1])
print("qKG public candidate:", kg_candidate)
print("qKG acquisition value:", kg_value)

## 9. Initialization axes must remain distinct

| Quantity | Initialization meaning |
| --- | --- |
| `num_restarts` | independent local-search starts |
| ordinary `q` | real joint candidate rows |
| `d` | current search-coordinate dimension |
| `X_pending` rows | acquisition context; not an IC axis |
| fantasy-model batch | model batch; not an IC axis |
| robust perturbation scenarios | uncertainty scenarios; not an IC axis |
| one-shot auxiliary rows | optimization variables; therefore augmented q |

## 10. Geometry-specific initialization

通常box search以外ではinitialization coordinate自体が変わります。

- REMBO / HeSBO: embedded coordinates
- ALEBO: feasible polytope sampling
- TuRBO: current trust-region bounds
- Hybrid: global-stage candidate becomes local explicit start
- Mixed one-shot: augmented rowsごとのcategorical assignment semantics

これらをpublic input-spaceのstandard box initializerへ機械的に押し込みません。

## 11. Constraint boundary

linear constraintsではBoTorchがfeasible startsを生成できます。nonlinear constraintsではfeasible `batch_initial_conditions`または適切な`ic_generator`が必要です。inter-point constraintsはjoint q relationなので、独立q=1 initializationへ分解できません。

## 12. Related documentation

- `docs/theory/optimization/03_initialization.md`
- `docs/optimization/initialization-compatibility.md`
- `docs/optimization/initialization.md`
- `31_acquisition_optimization.ipynb`
- `32_batch_async_fantasization.ipynb`

Initializationはoptimizerの付属パラメータではなく、search geometryとq semanticsを保つための契約です。